# Interactive Householder Reflection Visualizer

A Householder matrix $H = I - 2\mathbf{m}\mathbf{m}^\top$ (for unit vector $\mathbf{m}$) reflects vectors across the hyperplane orthogonal to $\mathbf{m}$.

More generally, the **delta rule** in Gated Delta Net uses $(I - \beta\,\mathbf{k}\mathbf{k}^\top)$ where:
- $\beta = 0$ → identity (no change)
- $\beta = 1$ → projection onto the plane orthogonal to $\mathbf{k}$
- $\beta = 2$ → full Householder reflection

Use the sliders below to drag **m** (by angle), **v** (by x/y), and **β** to see how they interact.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def draw_scene(ax, m_angle, v_x, v_y, beta):
    """Draw the full Householder scene on a given axes."""
    m = np.array([np.cos(np.radians(m_angle)), np.sin(np.radians(m_angle))])
    v = np.array([v_x, v_y])
    M = np.eye(2) - beta * np.outer(m, m)
    Mv = M @ v

    # Reflection plane
    perp = np.array([-m[1], m[0]])
    t = np.linspace(-3, 3, 200)
    ax.plot(perp[0]*t, perp[1]*t, "k--", lw=1.2, alpha=0.5, label="Plane orthogonal to m")

    # m vector
    ax.annotate("", xy=(m[0]*1.5, m[1]*1.5), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color="black", lw=2.5))
    ax.plot(m[0]*1.5, m[1]*1.5, "ko", ms=7, zorder=5)
    ax.text(m[0]*1.55+0.05, m[1]*1.55+0.05, r"$\mathbf{m}$", fontsize=14, fontweight="bold")

    # v vector
    ax.annotate("", xy=(v[0], v[1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color="tab:blue", lw=2.5))
    ax.plot(v[0], v[1], "o", color="tab:blue", ms=7, zorder=5)
    ax.text(v[0]+0.08, v[1]+0.08, r"$\mathbf{v}$", fontsize=13, color="tab:blue")

    # Transformed vector
    ax.annotate("", xy=(Mv[0], Mv[1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color="tab:red", lw=2.5, linestyle="--"))
    ax.text(Mv[0]+0.08, Mv[1]+0.08, r"$(I - \beta\, mm^\top)v$", fontsize=12, color="tab:red")

    # Connecting line
    ax.plot([v[0], Mv[0]], [v[1], Mv[1]], ":", color="grey", lw=0.8)

    # Info box
    beta_label = ""
    if abs(beta) < 0.01: beta_label = "  (identity)"
    elif abs(beta - 1.0) < 0.01: beta_label = "  (projection)"
    elif abs(beta - 2.0) < 0.01: beta_label = "  (Householder)"
    info = (f"||v||  = {np.linalg.norm(v):.2f}\n"
            f"||Mv|| = {np.linalg.norm(Mv):.2f}\n"
            f"\u03b2 = {beta:.2f}{beta_label}")
    ax.text(-2.8, -2.4, info, fontsize=10, family="monospace",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.7))

    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
    ax.set_title(r"$(I - \beta\, \mathbf{m}\mathbf{m}^\top)\, \mathbf{v}$", fontsize=15)
    ax.legend(loc="upper left", fontsize=10)

def update_householder(m_angle=45, v_x=1.5, v_y=0.5, beta=2.0):
    plt.close("all")
    fig, ax = plt.subplots(figsize=(7, 7))
    draw_scene(ax, m_angle, v_x, v_y, beta)
    plt.tight_layout()
    plt.show()

interact(
    update_householder,
    m_angle=FloatSlider(value=45, min=0, max=360, step=1, description="m angle (\u00b0)"),
    v_x=FloatSlider(value=1.5, min=-2.5, max=2.5, step=0.1, description="v_x"),
    v_y=FloatSlider(value=0.5, min=-2.5, max=2.5, step=0.1, description="v_y"),
    beta=FloatSlider(value=2.0, min=0.0, max=2.0, step=0.01, description="\u03b2"),
);

## Grid Reflection

See how the Householder matrix mirrors an entire grid of points. The color encodes original x-position so you can track where each point ends up.

In [ ]:
xs = np.linspace(-2, 2, 15)
ys = np.linspace(-2, 2, 15)
grid = np.array([[x, y] for x in xs for y in ys])

def update_grid(m_angle=45, beta=2.0):
    plt.close("all")
    m = np.array([np.cos(np.radians(m_angle)), np.sin(np.radians(m_angle))])
    M = np.eye(2) - beta * np.outer(m, m)
    transformed = grid @ M.T
    perp = np.array([-m[1], m[0]])
    t = np.linspace(-3, 3, 100)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, pts, title in [
        (axes[0], grid, "Original Grid"),
        (axes[1], transformed, f"After $(I - {beta:.1f}\\,mm^\\top)$"),
    ]:
        ax.scatter(pts[:, 0], pts[:, 1], s=14, c=grid[:, 0], cmap="coolwarm")
        ax.plot(perp[0]*t, perp[1]*t, "k--", lw=1.2, label="Plane ortho to m")
        ax.annotate("", xy=(m[0], m[1]), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color="black", lw=2))
        ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
        ax.set_aspect("equal"); ax.grid(True, alpha=0.3)
        ax.set_title(title, fontsize=13)
        ax.legend(loc="upper left")

    plt.tight_layout()
    plt.show()

interact(
    update_grid,
    m_angle=FloatSlider(value=45, min=0, max=360, step=1, description="m angle (\u00b0)"),
    beta=FloatSlider(value=2.0, min=0.0, max=2.0, step=0.01, description="\u03b2"),
);

## Key Properties

Three special cases that build intuition:
1. **Parallel to m** → vector gets negated (at β=2) or scaled down (β<2)
2. **Orthogonal to m** → vector is completely unchanged regardless of β
3. **General vector** → reflected across the hyperplane orthogonal to m

The norm is always preserved when β=2 (Householder is orthogonal). For other β values the norm can shrink.

In [ ]:
def update_properties(m_angle=45, beta=2.0):
    plt.close("all")
    m = np.array([np.cos(np.radians(m_angle)), np.sin(np.radians(m_angle))])
    M = np.eye(2) - beta * np.outer(m, m)

    v_parallel = m * 1.5
    v_perp = np.array([-m[1], m[0]]) * 1.5
    v_general = np.array([1.5, 0.3])

    cases = [
        (v_parallel, "Parallel to m"),
        (v_perp, "Orthogonal to m"),
        (v_general, "General vector"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (v, label) in zip(axes, cases):
        Mv = M @ v
        perp = np.array([-m[1], m[0]])
        t = np.linspace(-2.5, 2.5, 100)
        ax.plot(perp[0]*t, perp[1]*t, "k--", lw=1.2, label="Plane ortho to m")

        ax.annotate("", xy=(m[0], m[1]), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color="black", lw=2))
        ax.text(m[0]+0.05, m[1]+0.05, r"$\mathbf{m}$", fontsize=12)

        ax.annotate("", xy=(v[0], v[1]), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color="tab:blue", lw=2.5))
        ax.text(v[0]+0.05, v[1]+0.08, r"$\mathbf{v}$", fontsize=12, color="tab:blue")

        ax.annotate("", xy=(Mv[0], Mv[1]), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color="tab:red", lw=2.5, linestyle="--"))
        ax.text(Mv[0]+0.05, Mv[1]+0.08, r"$Mv$", fontsize=12, color="tab:red")

        ax.text(-2.3, -2.0,
                f"||v||  = {np.linalg.norm(v):.2f}\n||Mv|| = {np.linalg.norm(Mv):.2f}",
                fontsize=10, family="monospace",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

        ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
        ax.set_aspect("equal"); ax.grid(True, alpha=0.3)
        ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
        ax.set_title(label, fontsize=12)
        ax.legend(loc="upper left", fontsize=9)

    fig.suptitle(f"Properties at \u03b2 = {beta:.2f}", fontsize=14)
    plt.tight_layout()
    plt.show()

interact(
    update_properties,
    m_angle=FloatSlider(value=45, min=0, max=360, step=1, description="m angle (\u00b0)"),
    beta=FloatSlider(value=2.0, min=0.0, max=2.0, step=0.01, description="\u03b2"),
);